# 02 · Feature selection: what can be dropped?

**Goal:** remove features that carry no signal, so later steps (feature engineering, tuning, SHAP for the presentation) work with a cleaner, more stable set. Dropping noise rarely raises AUC much; the aim is not to *lose* any.

**Method** (all on the fixed folds in `common.py`: 5-fold × 3 seeds on train + valid):
1. **Group drop tests (LightGBM):** remove one group of candidate features at a time and compare to the full model **fold by fold**. A group is droppable if removing it doesn't hurt: Δ AUC ≥ −2 × its standard error.
2. **Null importance:** compare each feature's real LightGBM gain with its gain when the target is shuffled 30 times. Features whose real gain never beats the shuffled 75th percentile are likely noise. This is a cross-check on step 1.
3. **Combined check:** drop all droppable groups together and confirm on **both** LightGBM and CatBoost.
4. Save the kept list to `models/features_selected.json` for the next notebooks.

The candidate groups come from the EDA (`eda/eda.ipynb` §4, 5, 8–10).

In [1]:
import json, sys
import numpy as np, pandas as pd, lightgbm as lgb
sys.path.insert(0, '.')
import common as C

lab, hold = C.load()
F = C.base_features(lab)
res = {'full': C.cv_score('fs_full_lgb', lab, F, notes='feature selection: all features')}
print(f"full LightGBM: {res['full']['auc_mean']:.4f} ± {res['full']['auc_se']:.4f}, {len(F)} features")

full LightGBM: 0.6940 ± 0.0036, 77 features


## 1. Group drop tests
| Group | Why it's a candidate (EDA) |
|---|---|
| duplicates | ρ 0.93–0.999 with a twin that stays (`DRV_YOUNG_SHARE`, `DRV_TELEMATICS_ANY`, `ACS_COMMUTE_MEAN_MIN`) |
| missing flags | conversion within noise when missing, except `QUOTED_AMT_MISSING` (kept) |
| flat behavior | channel, multi-quote, follow-up days, parking and prior claims were flat alone *and* in interactions |
| flat vehicle | safety, garage, anti-theft, mileage, age and ownership counts were within noise |
| weak Census | income, education, median age and WFH had AUC ≈ 0.50–0.52 |
| date parts | no month or weekday pattern (`QUOTE_YEAR` is kept: holdout spans the same years) |
| policy vehicle counts | `TOTAL_NUM_VEH` + `NUM_*_VEH` from the policy table, which disagree with the vehicle table |
| vehicle-table counts | `VEH_N_ROWS` + `VEH_N_*`, the other side of that conflict |

In [2]:
GROUPS = {
    'duplicates':            ['DRV_DEPENDENT_SHARE', 'DRV_TELEMATICS_SHARE', 'ACS_COMMUTE_45PLUS'],
    'missing flags':         ['CREDIT_SCORE_MISSING', 'CAT_ZONE_MISSING', 'ZIP_CD_MISSING', 'ACS_ZIP3_FILLED'],
    'flat behavior':         ['QUOTE_CHANNEL', 'MULTI_QUOTE_IND', 'DAYS_TO_FOLLOWUP', 'PRIMARY_PARKING', 'PRIOR_CLAIMS_CNT'],
    'flat vehicle':          ['VEH_SAFETY_MIN', 'VEH_SAFETY_MEAN', 'VEH_GARAGE_SHARE', 'VEH_ANTI_THEFT_SHARE', 'VEH_MILEAGE_SUM',
                              'VEH_MILEAGE_MAX', 'VEH_AGE_MIN', 'VEH_AGE_MEAN', 'VEH_AGE_MAX', 'VEH_N_MAKES'],
    'weak Census':           ['ACS_MED_HH_INCOME', 'ACS_EDU_NO_HS', 'ACS_EDU_BACHELOR_PLUS', 'ACS_MED_AGE', 'ACS_COMMUTE_WFH'],
    'date parts':            ['QUOTE_MONTH', 'QUOTE_DOW'],
    'policy vehicle counts': ['TOTAL_NUM_VEH', 'NUM_OWNED_VEH', 'NUM_LEASED_VEH', 'NUM_LOANED_VEH'],
    'vehicle-table counts':  ['VEH_N_ROWS', 'VEH_N_OWNED', 'VEH_N_LEASED', 'VEH_N_LOANED'],
}
assert all(c in F for g in GROUPS.values() for c in g), 'unknown column in GROUPS'
for g, cols in GROUPS.items():
    res[g] = C.cv_score(f'fs_drop_{g}', lab, [c for c in F if c not in cols], notes=f'feature selection: drop {g}')

t = C.summary(list(res.values()), baseline='fs_full_lgb').rename(index=lambda n: n.removeprefix('fs_drop_'))
t['droppable'] = (t['Δ auc'] >= -2 * t['se Δ']) & (t.index != 'fs_full_lgb')
t.style.format({'auc_mean': '{:.4f}', 'auc_se': '{:.4f}', 'Δ auc': '{:+.4f}', 'se Δ': '{:.4f}', 'top_decile_prec': '{:.1%}'})

,model,n_feats,rounds,auc_mean,Δ auc,se Δ,auc_se,top_decile_prec,droppable
name,,,,,,,,,
fs_full_lgb,lgb,77,337,0.6940,+0.0000,0.0000,0.0036,27.8%,False
duplicates,lgb,74,386,0.6942,+0.0003,0.0005,0.0033,27.8%,True
missing flags,lgb,73,366,0.6937,-0.0003,0.0005,0.0033,27.7%,True
flat behavior,lgb,72,365,0.6943,+0.0003,0.0005,0.0035,27.7%,True
flat vehicle,lgb,67,319,0.6936,-0.0004,0.0005,0.0034,27.7%,True
weak Census,lgb,72,351,0.6944,+0.0005,0.0006,0.0035,27.8%,True
date parts,lgb,75,369,0.6944,+0.0005,0.0005,0.0034,27.8%,True
policy vehicle counts,lgb,73,278,0.6937,-0.0003,0.0004,0.0033,27.9%,True
vehicle-table counts,lgb,73,243,0.6929,-0.0011,0.0004,0.0033,27.9%,False


**Findings**
- **7 of 8 groups can be dropped:** removing them changes AUC by −0.0004 to +0.0005, all within about 1 standard error of zero.
- **The vehicle-table counts must stay:** dropping `VEH_N_ROWS` and related columns costs **−0.0011 (≈ 2.7 se)**, a real loss.
- **The policy-table counts (`TOTAL_NUM_VEH`, `NUM_*_VEH`) can go.** This settles the conflict from preprocessing: the vehicle table's count carries the signal and the policy table's doesn't.
- **The flat-behavior group can go,** confirming the EDA: channel, multi-quote, follow-up days, parking and prior claims add nothing even inside a model that can combine them.

## 2. Null importance
1. Fit LightGBM on all labeled rows with the full feature set and the CV-chosen number of trees, and record each feature's **gain**, averaged over 3 seeds.
2. Shuffle the target 30 times and refit each time to get each feature's gain under "no relationship".
3. A feature is **suspect** if its real gain is below the 75th percentile of its shuffled gains. Even in the best case it looks like noise.

High-cardinality numeric columns attract gain by chance, which is exactly what this test corrects for.

In [3]:
rounds, P = res['full']['rounds'], C.LGB_PARAMS
gain = lambda yv, seed: pd.Series(lgb.train(P | {'seed': seed}, lgb.Dataset(lab[F], yv), rounds).feature_importance('gain'), F)
real = pd.concat([gain(lab.CONVERT_IND, s) for s in range(3)], axis=1).mean(axis=1)
rng = np.random.default_rng(0)
null = pd.concat([gain(rng.permutation(lab.CONVERT_IND.values), 100 + i) for i in range(30)], axis=1)
ni = pd.DataFrame({'real gain': real, 'null p75': null.quantile(.75, axis=1), 'null max': null.max(axis=1)})
ni['real / null p75'] = ni['real gain'] / ni['null p75']
ni['suspect'] = ni['real gain'] < ni['null p75']
ni['in a droppable group'] = ni.index.isin([c for g in t.index[t.droppable] for c in GROUPS.get(g, [])])
print(f"suspect features: {ni.suspect.sum()} of {len(F)}")
ni.sort_values('real / null p75').head(30).round(2)

suspect features: 46 of 77


,real gain,null p75,null max,real / null p75,suspect,in a droppable group
DRV_MARRIED_SHARE,148.80,457.64,547.78,0.33,True,False
DRV_MALE_SHARE,150.26,445.65,570.80,0.34,True,False
PRIOR_CLAIMS_CNT,85.13,249.38,407.48,0.34,True,True
DRV_SENIOR_SHARE,26.00,66.65,120.31,0.39,True,False
MULTI_QUOTE_IND,55.17,134.11,211.23,0.41,True,True
VEH_N_LEASED,76.31,168.38,275.39,0.45,True,False
NUM_LEASED_VEH,81.48,172.49,341.66,0.47,True,True
VEH_GARAGE_SHARE,286.85,600.54,782.89,0.48,True,True
STATE_CD,157.82,326.81,526.43,0.48,True,False
QUOTE_MONTH,309.36,632.32,905.45,0.49,True,True


**Findings**
- **The test flags 46 of 77 features, which is too many to trust.** It flags `STATE_CD`, and in a re-run on the reduced set it nearly flags `CREDIT_SCORE` (ratio ≈ 1.0), both clearly predictive in the EDA.
- **Why:** gain spreads across correlated features (state ↔ Census, credit ↔ price), and continuous features pick up gain even on a shuffled target.
- **How it's used:** as a source of extra candidates, not a decision rule.
- **Most of the features it flags most strongly are already in the droppable groups.** The rest, mainly household shares (married, male, senior) and the vehicle-ownership split, are tested properly in §4.

## 3. Combined drop, confirmed on both models
- All droppable groups are removed together, and the reduced set is compared fold by fold against the full set, for LightGBM and CatBoost.
- If the combined drop hurts either model (Δ < −2 se), groups are added back one at a time, starting with the group whose removal cost the most in step 1.

In [4]:
DROP = [c for g in t.index[t.droppable] for c in GROUPS[g]]
KEEP = [c for c in F if c not in DROP]
print(f'dropping {len(DROP)} features from {len(t.index[t.droppable])} groups: {list(t.index[t.droppable])}')

res['reduced'] = C.cv_score('fs_reduced_lgb', lab, KEEP, notes='feature selection: combined drop')
cbr = {'full': C.cv_score('fs_full_cb', lab, F, model='cb', notes='feature selection: all features'),
       'reduced': C.cv_score('fs_reduced_cb', lab, KEEP, model='cb', notes='feature selection: combined drop')}
display(pd.concat([C.summary([res['full'], res['reduced']], baseline='fs_full_lgb'),
                   C.summary(list(cbr.values()), baseline='fs_full_cb')]).style.format(
    {'auc_mean': '{:.4f}', 'auc_se': '{:.4f}', 'Δ auc': '{:+.4f}', 'se Δ': '{:.4f}', 'top_decile_prec': '{:.1%}'}))

dropping 33 features from 7 groups: ['duplicates', 'missing flags', 'flat behavior', 'flat vehicle', 'weak Census', 'date parts', 'policy vehicle counts']


,model,n_feats,rounds,auc_mean,Δ auc,se Δ,auc_se,top_decile_prec
name,,,,,,,,
fs_full_lgb,lgb,77,337,0.6940,+0.0000,0.0000,0.0036,27.8%
fs_reduced_lgb,lgb,44,403,0.6941,+0.0002,0.0007,0.0036,27.9%
fs_full_cb,cb,77,449,0.6986,+0.0000,0.0000,0.0035,28.0%
fs_reduced_cb,cb,44,450,0.6992,+0.0006,0.0005,0.0034,28.2%


In [5]:
# Add groups back (most costly first) while the combined drop significantly hurts either model
hurts = lambda a, b: (a['fold_aucs'] - b['fold_aucs']).mean() < -2 * (a['fold_aucs'] - b['fold_aucs']).std() / np.sqrt(15)
order = t[t.droppable].sort_values('Δ auc').index.tolist()
while order and (hurts(res['reduced'], res['full']) or hurts(cbr['reduced'], cbr['full'])):
    g = order.pop(0); KEEP += GROUPS[g]; DROP = [c for c in DROP if c not in GROUPS[g]]
    print(f'adding back: {g}')
    res['reduced'] = C.cv_score(f'fs_reduced_lgb+{g}', lab, KEEP, notes=f'feature selection: add back {g}')
    cbr['reduced'] = C.cv_score(f'fs_reduced_cb+{g}', lab, KEEP, model='cb', notes=f'feature selection: add back {g}')
KEEP = [c for c in F if c in KEEP]  # restore original column order

## 4. Second round: features the null-importance test flagged
Null importance can't decide on its own: it also flags `CREDIT_SCORE`, which is clearly predictive, because continuous features pick up gain even on a shuffled target. Two groups it flagged that weren't tested in §1 *are* plausible noise, though, so they get the same paired drop test, **on both models**, against the reduced set:
- **household shares:** senior, married and male shares and telematics-any. All were flat in the EDA.
- **vehicle ownership split:** owned/leased/loaned counts from the vehicle table. `VEH_N_ROWS`, the total, stays: it's needed (§1).

A group is dropped only if neither model is hurt.

In [6]:
ROUND2 = {'household shares': ['DRV_SENIOR_SHARE', 'DRV_MARRIED_SHARE', 'DRV_MALE_SHARE', 'DRV_TELEMATICS_ANY'],
          'vehicle ownership split': ['VEH_N_OWNED', 'VEH_N_LEASED', 'VEH_N_LOANED']}
rows = []
for g, cols in ROUND2.items():
    trial = [c for c in KEEP if c not in cols]
    a = C.cv_score(f'fs_r2_{g}_lgb', lab, trial, notes=f'feature selection round 2: drop {g}')
    b = C.cv_score(f'fs_r2_{g}_cb', lab, trial, model='cb', notes=f'feature selection round 2: drop {g}')
    ok = not hurts(a, res['reduced']) and not hurts(b, cbr['reduced'])
    rows += [C.summary([res['reduced'], a], baseline=res['reduced']['name']).iloc[[1]].assign(dropped=ok),
             C.summary([cbr['reduced'], b], baseline=cbr['reduced']['name']).iloc[[1]].assign(dropped=ok)]
    if ok:
        KEEP, DROP, res['reduced'], cbr['reduced'] = trial, DROP + cols, a, b
display(pd.concat(rows).style.format({'auc_mean': '{:.4f}', 'auc_se': '{:.4f}', 'Δ auc': '{:+.4f}', 'se Δ': '{:.4f}', 'top_decile_prec': '{:.1%}'}))

json.dump({'features': KEEP, 'dropped': DROP}, open('features_selected.json', 'w'), indent=1)
print(f'saved {len(KEEP)} features (dropped {len(DROP)}) to models/features_selected.json')
print(f"final reduced set: LightGBM {res['reduced']['auc_mean']:.4f} (full {res['full']['auc_mean']:.4f}) | "
      f"CatBoost {cbr['reduced']['auc_mean']:.4f} (full {cbr['full']['auc_mean']:.4f})")

,model,n_feats,rounds,auc_mean,Δ auc,se Δ,auc_se,top_decile_prec,dropped
name,,,,,,,,,
fs_r2_household shares_lgb,lgb,40,346,0.6955,+0.0013,0.0005,0.0034,27.9%,True
fs_r2_household shares_cb,cb,40,427,0.6998,+0.0006,0.0007,0.0034,27.9%,True
fs_r2_vehicle ownership split_lgb,lgb,37,369,0.6956,+0.0002,0.0004,0.0033,27.8%,True
fs_r2_vehicle ownership split_cb,cb,37,424,0.7011,+0.0013,0.0008,0.0032,28.3%,True


saved 37 features (dropped 40) to models/features_selected.json
final reduced set: LightGBM 0.6956 (full 0.6940) | CatBoost 0.7011 (full 0.6986)


**Findings**
- Dropping all 7 groups at once (33 features) **doesn't hurt either model**: LightGBM +0.0002, CatBoost +0.0006 (both within noise). No group needed to be added back.

**Round-2 findings**
- **Household shares:** dropping them *helps* LightGBM (**+0.0013, ≈ 2.6 se**) and is neutral for CatBoost (+0.0006). They were adding noise.
- **Vehicle ownership split:** neutral for LightGBM (+0.0002) and slightly positive for CatBoost (+0.0013, ≈ 1.6 se).

## Summary
| | All 77 features | **Selected 37** | Change |
|---|---|---|---|
| LightGBM CV AUC | 0.6940 | **0.6956** | +0.0016 |
| CatBoost CV AUC | 0.6986 | **0.7011** | +0.0025 |

- **Over half the features are dropped** (40 of 77) with a small *gain* on both models. Top-decile precision is unchanged (~28%).
- **What remains:**
  - price, credit, discount and home policy
  - coverage package, CAT zone and prior carrier
  - number of drivers and the driver age / education / safety / accident / commute summaries
  - vehicle count and the luxury flag
  - the urban Census features
  - state and quote year
- `models/features_selected.json` is the input to the next notebook, feature engineering (plan Step 2).